In [1]:
!pip install -q kaggle pandas numpy scikit-learn scipy joblib nltk rouge-score

  Preparing metadata (setup.py) ... done


In [2]:
from google.colab import files

uploaded = files.upload()

Saving kaggle.json to kaggle.json


In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API configured.")

Kaggle API configured.


In [4]:
!mkdir -p /content/race_kaggle
!kaggle datasets download -d ankitdhiman7/race-dataset -p /content/race_kaggle --unzip

Dataset URL: https://www.kaggle.com/datasets/ankitdhiman7/race-dataset
License(s): unknown
100% 70.0M/70.0M [00:03<00:00, 23.9MB/s]



In [5]:
import os

for root, dirs, files in os.walk("/content/race_kaggle"):
    level = root.replace("/content/race_kaggle", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files[:10]:
        print(f"{indent}  {f}")

race_kaggle/
  test.csv
  train.csv
  dev.csv


In [6]:
import os
import json
import pandas as pd

BASE_PATH = "/content/race_kaggle"

def find_csv_files(base_path):
    csvs = []
    for root, _, files in os.walk(base_path):
        for f in files:
            if f.lower().endswith(".csv"):
                csvs.append(os.path.join(root, f))
    return csvs

def load_csv_splits(base_path):
    csvs = find_csv_files(base_path)
    print("CSV files found:")
    for c in csvs:
        print(c)

    train_path = None
    val_path = None
    test_path = None

    for path in csvs:
        name = os.path.basename(path).lower()
        if "train" in name:
            train_path = path
        elif "val" in name or "dev" in name:
            val_path = path
        elif "test" in name:
            test_path = path

    if train_path is None or test_path is None:
        return None, None, None

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    if val_path is not None:
        val_df = pd.read_csv(val_path)
    else:
        val_df = test_df.sample(frac=0.5, random_state=42)
        test_df = test_df.drop(val_df.index)

    return train_df, val_df, test_df


def load_race_txt_split(split_path):
    rows = []

    for root, _, files in os.walk(split_path):
        for file in files:
            if file.endswith(".txt"):
                full_path = os.path.join(root, file)

                with open(full_path, "r", encoding="utf-8") as f:
                    sample = json.load(f)

                article = sample["article"]
                questions = sample["questions"]
                options = sample["options"]
                answers = sample["answers"]

                for i, q in enumerate(questions):
                    opts = options[i]

                    rows.append({
                        "id": f"{file}_q{i}",
                        "article": article,
                        "question": q,
                        "A": opts[0],
                        "B": opts[1],
                        "C": opts[2],
                        "D": opts[3],
                        "answer": answers[i]
                    })

    return pd.DataFrame(rows)


def find_split_folder(base_path, split_name):
    candidates = []

    for root, dirs, files in os.walk(base_path):
        folder_name = os.path.basename(root).lower()
        if folder_name == split_name.lower():
            candidates.append(root)

    return candidates[0] if candidates else None


def load_kaggle_race(base_path):
    train_df, val_df, test_df = load_csv_splits(base_path)

    if train_df is not None:
        print("Loaded CSV version.")
        return train_df, val_df, test_df

    print("CSV not found. Trying original RACE TXT format...")

    train_folder = find_split_folder(base_path, "train")
    dev_folder = find_split_folder(base_path, "dev")
    test_folder = find_split_folder(base_path, "test")

    if train_folder is None or test_folder is None:
        raise FileNotFoundError("Could not find train/dev/test folders.")

    train_df = load_race_txt_split(train_folder)
    val_df = load_race_txt_split(dev_folder) if dev_folder else None
    test_df = load_race_txt_split(test_folder)

    if val_df is None:
        val_df = test_df.sample(frac=0.5, random_state=42)
        test_df = test_df.drop(val_df.index)

    print("Loaded TXT/JSON version.")
    return train_df, val_df, test_df


train_df, val_df, test_df = load_kaggle_race(BASE_PATH)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)
print(train_df.columns)
train_df.head()

CSV files found:
/content/race_kaggle/test.csv
/content/race_kaggle/train.csv
/content/race_kaggle/dev.csv
Loaded CSV version.
Train: (87866, 9)
Val: (87866, 9)
Test: (87866, 9)
Index(['Unnamed: 0', 'id', 'article', 'question', 'A', 'B', 'C', 'D',
       'answer'],
      dtype='object')


,Unnamed: 0,id,article,question,A,B,C,D,answer
0,0,middle7348.txt,In the summer between my first year and second...,Before the writer came to the high school summ...,instructor,camper,student,reporter,C
1,1,middle7348.txt,In the summer between my first year and second...,How many times did the writer invite the boy t...,Once,Twice,Three times,Many times,B
2,2,middle4305.txt,A bumpkin went to a big city for the first t...,The bumpkin thought _ .,his wife was as beautiful as the young girl,his wife was more beautiful than the short fat...,the short fat woman changed to a young girl in...,He should also buy a room of that kind for his...,C
3,3,middle4305.txt,A bumpkin went to a big city for the first t...,The room he saw was perhaps _ .,an office,a toilet,a lift,a helicopter,C
4,4,middle4305.txt,A bumpkin went to a big city for the first t...,We can see that the bumpkin had no knowledge ...,farming or gardening,how to do farm work,how to live in a city,modern city life,D


In [7]:
required_cols = ["id", "article", "question", "A", "B", "C", "D", "answer"]

print(train_df.columns)

# If id is missing, create it
for df_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    if "id" not in df.columns:
        df["id"] = [f"{df_name}_{i}" for i in range(len(df))]

# Keep only required columns
train_df = train_df[required_cols]
val_df = val_df[required_cols]
test_df = test_df[required_cols]

print(train_df.shape, val_df.shape, test_df.shape)
train_df.head()

Index(['Unnamed: 0', 'id', 'article', 'question', 'A', 'B', 'C', 'D',
       'answer'],
      dtype='object')
(87866, 8) (87866, 8) (87866, 8)


,id,article,question,A,B,C,D,answer
0,middle7348.txt,In the summer between my first year and second...,Before the writer came to the high school summ...,instructor,camper,student,reporter,C
1,middle7348.txt,In the summer between my first year and second...,How many times did the writer invite the boy t...,Once,Twice,Three times,Many times,B
2,middle4305.txt,A bumpkin went to a big city for the first t...,The bumpkin thought _ .,his wife was as beautiful as the young girl,his wife was more beautiful than the short fat...,the short fat woman changed to a young girl in...,He should also buy a room of that kind for his...,C
3,middle4305.txt,A bumpkin went to a big city for the first t...,The room he saw was perhaps _ .,an office,a toilet,a lift,a helicopter,C
4,middle4305.txt,A bumpkin went to a big city for the first t...,We can see that the bumpkin had no knowledge ...,farming or gardening,how to do farm work,how to live in a city,modern city life,D


In [8]:
print("Answer distribution:")
print(train_df["answer"].value_counts().sort_index())

train_df["article_word_count"] = train_df["article"].apply(lambda x: len(str(x).split()))
train_df["question_word_count"] = train_df["question"].apply(lambda x: len(str(x).split()))

print(train_df[["article_word_count", "question_word_count"]].describe())

Answer distribution:
answer
A    19146
B    22726
C    23891
D    22103
Name: count, dtype: int64
       article_word_count  question_word_count
count        87866.000000         87866.000000
mean           274.983702            10.008786
std             97.881345             3.382040
min              2.000000             1.000000
25%            217.000000             8.000000
50%            279.000000            10.000000
75%            326.000000            12.000000
max           1162.000000            63.000000


In [9]:
def build_verification_dataset(df):
    rows = []

    for _, row in df.iterrows():
        article = str(row["article"])
        question = str(row["question"])
        correct_answer = row["answer"]

        for option_label in ["A", "B", "C", "D"]:
            option_text = str(row[option_label])

            rows.append({
                "id": row["id"],
                "article": article,
                "question": question,
                "option_label": option_label,
                "option_text": option_text,
                "combined_text": article + " " + article + " " + question + " " + option_text,
                "is_correct": 1 if option_label == correct_answer else 0
            })

    return pd.DataFrame(rows)


train_verif = build_verification_dataset(train_df)
val_verif = build_verification_dataset(val_df)
test_verif = build_verification_dataset(test_df)

print(train_verif.shape)
print(train_verif["is_correct"].value_counts())

(351464, 7)
is_correct
0    263598
1     87866
Name: count, dtype: int64


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words="english",
    sublinear_tf=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

X_train = vectorizer.fit_transform(train_verif["combined_text"])
X_val = vectorizer.transform(val_verif["combined_text"])
X_test = vectorizer.transform(test_verif["combined_text"])

y_train = train_verif["is_correct"]
y_val = val_verif["is_correct"]
y_test = test_verif["is_correct"]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (351464, 10000)
X_val: (351464, 10000)
X_test: (351464, 10000)


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

logreg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="liblinear",
    random_state=42
)

svm = LinearSVC(
    class_weight="balanced",
    random_state=42,
    max_iter=3000
)

logreg.fit(X_train, y_train)
svm.fit(X_train, y_train)

print("Models trained.")

Models trained.


In [12]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd
import numpy as np

def evaluate_model(model, X, y, name):
    preds = model.predict(X)

    return {
        "dataset": "Kaggle RACE",
        "model": name,
        "accuracy": accuracy_score(y, preds),
        "precision": precision_score(y, preds, zero_division=0),
        "recall": recall_score(y, preds, zero_division=0),
        "macro_f1": f1_score(y, preds, average="macro"),
        "confusion_matrix": confusion_matrix(y, preds).tolist()
    }

logreg_results = evaluate_model(logreg, X_test, y_test, "Logistic Regression")
svm_results = evaluate_model(svm, X_test, y_test, "Linear SVM")

pd.DataFrame([logreg_results, svm_results])

,dataset,model,accuracy,precision,recall,macro_f1,confusion_matrix
0,Kaggle RACE,Logistic Regression,0.514329,0.259570,0.508866,0.479147,"[[136056, 127542], [43154, 44712]]"
1,Kaggle RACE,Linear SVM,0.514969,0.259235,0.506123,0.479248,"[[136522, 127076], [43395, 44471]]"


In [13]:
def hard_voting_ensemble_predict(models, X):
    all_preds = []

    for model in models:
        preds = model.predict(X)
        all_preds.append(preds)

    all_preds = np.array(all_preds)

    ensemble_preds = []

    for i in range(all_preds.shape[1]):
        votes = all_preds[:, i]
        final_vote = np.bincount(votes).argmax()
        ensemble_preds.append(final_vote)

    return np.array(ensemble_preds)


ensemble_preds = hard_voting_ensemble_predict([logreg, svm], X_test)

ensemble_results = {
    "dataset": "Kaggle RACE",
    "model": "Hard Voting Ensemble",
    "accuracy": accuracy_score(y_test, ensemble_preds),
    "precision": precision_score(y_test, ensemble_preds, zero_division=0),
    "recall": recall_score(y_test, ensemble_preds, zero_division=0),
    "macro_f1": f1_score(y_test, ensemble_preds, average="macro"),
    "confusion_matrix": confusion_matrix(y_test, ensemble_preds).tolist()
}

classification_results = pd.DataFrame([
    logreg_results,
    svm_results,
    ensemble_results
])

classification_results

,dataset,model,accuracy,precision,recall,macro_f1,confusion_matrix
0,Kaggle RACE,Logistic Regression,0.514329,0.259570,0.508866,0.479147,"[[136056, 127542], [43154, 44712]]"
1,Kaggle RACE,Linear SVM,0.514969,0.259235,0.506123,0.479248,"[[136522, 127076], [43395, 44471]]"
2,Kaggle RACE,Hard Voting Ensemble,0.533030,0.260215,0.470910,0.487662,"[[145964, 117634], [46489, 41377]]"


In [14]:
!pip install -q rouge-score nltk

In [15]:
import nltk
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

nltk.download("wordnet")
nltk.download("omw-1.4")

def generate_hints_simple(article, question):
    sentences = str(article).split(".")

    ranked = sorted(
        sentences,
        key=lambda x: len(set(x.lower().split()) & set(str(question).lower().split())),
        reverse=True
    )

    hints = [s.strip() for s in ranked[:3] if len(s.strip()) > 0]

    if len(hints) == 0:
        hints = [str(article)[:200]]

    return hints


scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rougeL"],
    use_stemmer=True
)

bleu_scores = []
meteor_scores = []
rouge1_scores = []
rougeL_scores = []

sample_eval = test_df.sample(min(100, len(test_df)), random_state=42)

for _, row in sample_eval.iterrows():
    article = str(row["article"])
    question = str(row["question"])

    generated_hints = generate_hints_simple(article, question)
    generated_text = " ".join(generated_hints)

    reference = article

    bleu = sentence_bleu(
        [reference.split()],
        generated_text.split()
    )

    meteor = meteor_score(
        [reference.split()],
        generated_text.split()
    )

    rouge_scores = scorer.score(reference, generated_text)

    bleu_scores.append(bleu)
    meteor_scores.append(meteor)
    rouge1_scores.append(rouge_scores["rouge1"].fmeasure)
    rougeL_scores.append(rouge_scores["rougeL"].fmeasure)

generation_results = pd.DataFrame([{
    "dataset": "Kaggle RACE",
    "BLEU": sum(bleu_scores) / len(bleu_scores),
    "METEOR": sum(meteor_scores) / len(meteor_scores),
    "ROUGE-1": sum(rouge1_scores) / len(rouge1_scores),
    "ROUGE-L": sum(rougeL_scores) / len(rougeL_scores)
}])

generation_results

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


,dataset,BLEU,METEOR,ROUGE-1,ROUGE-L
0,Kaggle RACE,0.048616,0.193413,0.362164,0.299377


In [16]:
tfds_generation_results = pd.DataFrame([{
    "dataset": "TFDS RACE",
    "BLEU": 0.0518,
    "METEOR": 0.1980,
    "ROUGE-1": 0.3644,
    "ROUGE-L": 0.3067
}])

generation_comparison = pd.concat(
    [tfds_generation_results, generation_results],
    ignore_index=True
)

generation_comparison

,dataset,BLEU,METEOR,ROUGE-1,ROUGE-L
0,TFDS RACE,0.051800,0.198000,0.364400,0.306700
1,Kaggle RACE,0.048616,0.193413,0.362164,0.299377


In [17]:
tfds_classification_results = pd.DataFrame([
    {
        "dataset": "TFDS RACE",
        "model": "Logistic Regression",
        "accuracy": 0.507,
        "precision": 0.256,
        "recall": 0.510,
        "macro_f1": 0.474
    },
    {
        "dataset": "TFDS RACE",
        "model": "Linear SVM",
        "accuracy": 0.501,
        "precision": 0.255,
        "recall": 0.518,
        "macro_f1": 0.470
    },
    {
        "dataset": "TFDS RACE",
        "model": "Hard Voting Ensemble",
        "accuracy": 0.541,
        "precision": 0.256,
        "recall": 0.439,
        "macro_f1": 0.488
    }
])

classification_comparison = pd.concat(
    [
        tfds_classification_results,
        classification_results.drop(columns=["confusion_matrix"], errors="ignore")
    ],
    ignore_index=True
)

classification_comparison

,dataset,model,accuracy,precision,recall,macro_f1
0,TFDS RACE,Logistic Regression,0.507000,0.256000,0.510000,0.474000
1,TFDS RACE,Linear SVM,0.501000,0.255000,0.518000,0.470000
2,TFDS RACE,Hard Voting Ensemble,0.541000,0.256000,0.439000,0.488000
3,Kaggle RACE,Logistic Regression,0.514329,0.259570,0.508866,0.479147
4,Kaggle RACE,Linear SVM,0.514969,0.259235,0.506123,0.479248
5,Kaggle RACE,Hard Voting Ensemble,0.533030,0.260215,0.470910,0.487662


In [18]:
classification_comparison.to_csv("classification_comparison_tfds_vs_kaggle.csv", index=False)
generation_comparison.to_csv("generation_comparison_tfds_vs_kaggle.csv", index=False)

print("Saved comparison CSV files.")

Saved comparison CSV files.


In [19]:
from google.colab import files

files.download("classification_comparison_tfds_vs_kaggle.csv")
files.download("generation_comparison_tfds_vs_kaggle.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>